# Views e Triggers: Lógica que Vive no Banco
---

## Contexto: o mesmo JOIN em 10 notebooks diferentes

O time de analytics criou 10 notebooks diferentes e todos repetem o mesmo JOIN quádruplo de clientes, pedidos, itens e produtos. Quando uma coluna foi renomeada no banco, quebraram os 10 de uma vez.

Ao mesmo tempo, o campo `valor_total` em `tb_pedidos` está sempre desatualizado porque depende do código Python lembrar de recalcular após cada alteração.

Existem dois recursos do banco que resolvem exatamente esses dois problemas: **Views** e **Triggers**.

Nesta aula você vai:

- Criar views para encapsular JOINs complexos e reutilizá-los como tabelas
- Consultar views de três formas diferentes
- Criar triggers para manter dados consistentes automaticamente
- Entender quando (e quando não) usar cada um

---

## 1. Configuração

In [13]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect, text,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_, Table, MetaData
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload, aliased
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


In [2]:
# Verificar tabelas com SQLAlchemy
insp = inspect(engine)

print(insp.get_table_names())

['tb_clientes', 'tb_itens_pedidos', 'tb_pedidos', 'tb_produtos']


## 2. O problema que Views resolvem

Sem view, cada notebook repete o mesmo JOIN:

```
Notebook 1:  JOIN clientes + JOIN itens + JOIN produtos  →  duplicação
Notebook 2:  JOIN clientes + JOIN itens + JOIN produtos  →  duplicação
Notebook 3:  JOIN clientes + JOIN itens + JOIN produtos  →  duplicação
```

Com view:
```
Notebook 1:  SELECT * FROM vw_vendas_detalhadas  →  reutilização
Notebook 2:  SELECT * FROM vw_vendas_detalhadas  →  reutilização
Notebook 3:  SELECT * FROM vw_vendas_detalhadas  →  reutilização
```

Uma **View** é uma consulta salva no banco que se comporta como uma tabela pronta para leitura.  

Quando alteramos a lógica em **um único lugar** todos os notebooks se beneficiam automaticamente.

## 3. Criando Views

O SQLAlchemy não tem um método ORM dedicado para criar views, isso é DDL (estrutura do banco), não dado.  
Usamos SQL nativo via `text()`, que é a abordagem correta aqui.

> **Views criam objetos permanentes no banco.** Se quiser experimentar sem alterar o banco principal, faça uma cópia: `cp database/datavendas.db database/datavendas_dev.db`

In [3]:
# View 1: um item por linha — ideal para análise de produtos
create_view_detalhada = """
CREATE VIEW vw_vendas_detalhadas AS
SELECT
    i.id                              AS item_id,
    p.id                              AS pedido_id,
    p.data_pedido,
    p.status,
    c.id                              AS cliente_id,
    c.nome                            AS cliente_nome,
    c.estado                          AS cliente_estado,
    prod.id                           AS produto_id,
    prod.nome                         AS produto_nome,
    prod.categoria                    AS produto_categoria,
    i.quantidade,
    i.preco_venda,
    (i.quantidade * i.preco_venda)     AS subtotal_item
FROM tb_pedidos p
JOIN tb_clientes       c    ON p.cliente_id  = c.id
JOIN tb_itens_pedidos  i    ON p.id          = i.pedido_id
JOIN tb_produtos       prod ON i.produto_id  = prod.id;
"""

# engine.begin() abre uma transação — tudo dentro do bloco é confirmado ao sair
with engine.begin() as conn:
    # No SQLite não existe CREATE OR REPLACE VIEW — o padrão é sempre DROP + CREATE
    conn.execute(text("DROP VIEW IF EXISTS vw_vendas_detalhadas"))
    conn.execute(text(create_view_detalhada))

print("View 'vw_vendas_detalhadas' criada ✅")

View 'vw_vendas_detalhadas' criada ✅


In [4]:
# View 2: um pedido por linha — ideal para métricas de receita
create_view_resumo = """
CREATE VIEW vw_pedidos_resumo AS
SELECT
    p.id                                           AS pedido_id,
    c.nome                                         AS cliente_nome,
    c.estado                                       AS cliente_estado,
    p.data_pedido,
    p.status,
    COALESCE(SUM(i.quantidade * i.preco_venda), 0)  AS total_calculado
FROM tb_pedidos p
JOIN      tb_clientes      c ON p.cliente_id = c.id
LEFT JOIN tb_itens_pedidos i ON p.id         = i.pedido_id
GROUP BY p.id, c.nome, c.estado, p.data_pedido, p.status;
"""

with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS vw_pedidos_resumo"))
    conn.execute(text(create_view_resumo))

print("View 'vw_pedidos_resumo' criada ✅")
print()
print("Diferença entre as duas views:")
print("  vw_vendas_detalhadas → 1 linha por ITEM   (análise de produto)")
print("  vw_pedidos_resumo    → 1 linha por PEDIDO (métricas de receita)")

View 'vw_pedidos_resumo' criada ✅

Diferença entre as duas views:
  vw_vendas_detalhadas → 1 linha por ITEM   (análise de produto)
  vw_pedidos_resumo    → 1 linha por PEDIDO (métricas de receita)


## 4. Consultando Views — 3 formas

Uma vez criada, a view se comporta como uma tabela normal para leitura.

### Forma 1: SQL nativo com `text()`

A mais simples, ideal para consultas rápidas e exportação para Pandas.

In [5]:
# SQL direto — trata a view como qualquer tabela
query = text("""
    SELECT pedido_id, cliente_nome, produto_nome, subtotal_item
    FROM vw_vendas_detalhadas
    WHERE cliente_estado = :estado
    ORDER BY subtotal_item DESC
    LIMIT 5
""")

with engine.connect() as conn:
    rows = conn.execute(query, {"estado": "SP"}).fetchall()

print("Top 5 itens vendidos em SP:")
if rows:
    for r in rows:
        print(f"  Pedido {r.pedido_id} | {r.cliente_nome} | {r.produto_nome} | R$ {r.subtotal_item}")
else:
    print("  (Sem dados — popule o banco para ver resultados)")

Top 5 itens vendidos em SP:
  Pedido 13 | Dr. Bryan Carvalho | Iste vitae delectus | R$ 42582.96
  Pedido 13 | Dr. Bryan Carvalho | Aperiam voluptas veniam | R$ 41702.13
  Pedido 91 | Srta. Julia Cassiano | Aut porro aspernatur | R$ 37340.100000000006
  Pedido 3 | Laís Santos | Neque aliquam | R$ 27865.4
  Pedido 100 | Srta. Julia Cassiano | Totam non | R$ 27214.739999999998


### Forma 2: Reflexão Core com `Table(autoload_with=engine)`

O SQLAlchemy lê a estrutura da view direto do banco e cria um objeto Python com as colunas via `.c`.  
Você ganha a sintaxe fluente do `select()` sem precisar definir nada manualmente.

In [8]:
# autoload_with lê a estrutura da view diretamente do banco
vw_resumo = Table("vw_pedidos_resumo", MetaData(), autoload_with=engine)

# A partir daqui, use .c para acessar as colunas — igual a uma tabela normal
stmt = (
    select(
        vw_resumo.c.pedido_id,
        vw_resumo.c.cliente_nome,
        vw_resumo.c.total_calculado,
    )
    .order_by(vw_resumo.c.total_calculado.desc())
    .limit(5)
)

with Session(engine) as session:
    resultados = session.execute(stmt).all()

print("Top 5 pedidos por receita (via reflexão Core):")
if resultados:
    for r in resultados:
        print(f"  Pedido {r.pedido_id} | {r.cliente_nome} | R$ {float(r.total_calculado):.2f}")
else:
    print("  (Sem dados)")

Top 5 pedidos por receita (via reflexão Core):
  Pedido 46 | Maria Sophia Guerra | R$ 137396.68
  Pedido 13 | Dr. Bryan Carvalho | R$ 112483.30
  Pedido 3 | Laís Santos | R$ 110713.96
  Pedido 59 | Dr. Diogo Sousa | R$ 103945.50
  Pedido 38 | Mathias Casa Grande | R$ 97986.41


### Forma 3: Classe ORM mapeada para a view

Você define uma classe Python apontando para a view exatamente como um modelo ORM normal.  
Use quando quiser acesso por atributo e integração com o restante do modelo.

> **Importante:** views são **somente leitura**. Nunca faça `session.add()` ou `session.delete()` em objetos desta classe, escreva sempre nas tabelas originais.

In [10]:
class VendaDetalhada(Base):
    __tablename__  = "vw_vendas_detalhadas"
    __table_args__ = {"extend_existing": True}

    # O ORM precisa de uma coluna que identifique cada linha de forma única
    item_id:           Mapped[int]     = mapped_column(Integer, primary_key=True)
    pedido_id:         Mapped[int]     = mapped_column(Integer)
    cliente_nome:      Mapped[str]     = mapped_column(String)
    cliente_estado:    Mapped[str]     = mapped_column(String)
    produto_nome:      Mapped[str]     = mapped_column(String)
    produto_categoria: Mapped[str]     = mapped_column(String)
    subtotal_item:     Mapped[Decimal] = mapped_column(Numeric)


# Usando com sintaxe ORM completa, agrupa receita por cliente
receita = func.sum(VendaDetalhada.subtotal_item).label("receita")

stmt = (
    select(
        VendaDetalhada.cliente_nome,
        VendaDetalhada.cliente_estado,
        receita,
    )
    .group_by(VendaDetalhada.cliente_nome, VendaDetalhada.cliente_estado)
    .order_by(receita.desc())
    .limit(5)
)

with Session(engine) as session:
    resultados = session.execute(stmt).all()

print("Top 5 clientes por receita (via ORM mapeado para view):")
print("-" * 80)
if resultados:
    for r in resultados:
        print(f"  {r.cliente_nome} ({r.cliente_estado}): R$ {float(r.receita):.2f}")
else:
    print("  (Sem dados)")

Top 5 clientes por receita (via ORM mapeado para view):
--------------------------------------------------------------------------------
  Maria Sophia Guerra (RJ): R$ 422479.56
  Théo Cirino (CE): R$ 412942.75
  Stella Guerra (DF): R$ 359972.12
  Luiz Miguel Fogaça (GO): R$ 333001.78
  Mathias Casa Grande (BA): R$ 329982.24


/tmp/ipykernel_161738/2461891961.py:1: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.VendaDetalhada, and will be replaced in the string-lookup table.
  class VendaDetalhada(Base):


### Qual forma usar?

| Forma | Quando usar |
|---|---|
| `text()` — SQL nativo | Consultas rápidas, exportar para Pandas, scripts pontuais |
| `Table(autoload_with=)` | Quer `select()` fluente sem definir colunas à mão |
| Classe ORM mapeada | Quer acesso por atributo, integração forte com outros modelos |

**Regra prática:** comece com `Table(autoload_with=)`. Use a classe ORM só se precisar de métodos customizados.

---

## 5. O problema que Triggers resolvem

O campo `valor_total` em `tb_pedidos` deveria ser sempre igual à soma dos itens.  
Mas hoje esse cálculo vive no código Python, o que significa:

- Se alguém inserir um item direto no banco, `valor_total` fica desatualizado
- Se o código tiver um bug no cálculo, os dados ficam errados silenciosamente

Com um **Trigger**, o banco recalcula automaticamente a cada inserção, atualização ou remoção:

```
Sem trigger:                     Com trigger:
──────────────────────────       ───────────────────────────────────────
Python insere item            →  Python insere item
Python calcula valor_total    →  Banco atualiza valor_total sozinho ✅
Python salva valor_total      →
```

## 6. Anatomia de um Trigger no SQLite

```sql
CREATE TRIGGER nome_do_trigger
AFTER INSERT ON tabela_monitorada    ← quando dispara
BEGIN
    UPDATE outra_tabela SET coluna = (
        SELECT SUM(...) FROM tabela_monitorada
        WHERE id = NEW.pedido_id     ← NEW = linha que acabou de ser inserida
    );
END;
```

- `AFTER INSERT` — dispara depois de cada `INSERT` na tabela monitorada
- `NEW` — linha que acabou de ser inserida; para `DELETE` existe também `OLD`
- O trigger roda **dentro da mesma transação**: se ele falhar, o INSERT falha junto

## 7. Criando Triggers para manter `valor_total` atualizado

In [11]:
# Trigger 1: dispara após INSERT de um item
trigger_insert = """
CREATE TRIGGER IF NOT EXISTS atualiza_total_insert
AFTER INSERT ON tb_itens_pedidos
BEGIN
    UPDATE tb_pedidos
    SET valor_total = (
        SELECT COALESCE(SUM(quantidade * preco_venda), 0)
        FROM tb_itens_pedidos
        WHERE pedido_id = NEW.pedido_id
    )
    WHERE id = NEW.pedido_id;
END;
"""

# Trigger 2: dispara após UPDATE de um item
# Recalcula o pedido atual (NEW) e, se o item mudou de pedido, recalcula o anterior (OLD)
trigger_update = """
CREATE TRIGGER IF NOT EXISTS atualiza_total_update
AFTER UPDATE ON tb_itens_pedidos
BEGIN
    UPDATE tb_pedidos
    SET valor_total = (
        SELECT COALESCE(SUM(quantidade * preco_venda), 0)
        FROM tb_itens_pedidos
        WHERE pedido_id = NEW.pedido_id
    )
    WHERE id = NEW.pedido_id;

    UPDATE tb_pedidos
    SET valor_total = (
        SELECT COALESCE(SUM(quantidade * preco_venda), 0)
        FROM tb_itens_pedidos
        WHERE pedido_id = OLD.pedido_id
    )
    WHERE id = OLD.pedido_id AND OLD.pedido_id != NEW.pedido_id;
END;
"""

# Trigger 3: dispara após DELETE de um item
# OLD.pedido_id = ID do pedido que perdeu o item
trigger_delete = """
CREATE TRIGGER IF NOT EXISTS atualiza_total_delete
AFTER DELETE ON tb_itens_pedidos
BEGIN
    UPDATE tb_pedidos
    SET valor_total = (
        SELECT COALESCE(SUM(quantidade * preco_venda), 0)
        FROM tb_itens_pedidos
        WHERE pedido_id = OLD.pedido_id
    )
    WHERE id = OLD.pedido_id;
END;
"""

with engine.begin() as conn:
    # Remove versões anteriores antes de recriar
    conn.execute(text("DROP TRIGGER IF EXISTS atualiza_total_insert"))
    conn.execute(text("DROP TRIGGER IF EXISTS atualiza_total_update"))
    conn.execute(text("DROP TRIGGER IF EXISTS atualiza_total_delete"))
    conn.execute(text(trigger_insert))
    conn.execute(text(trigger_update))
    conn.execute(text(trigger_delete))

print("Triggers criados: INSERT, UPDATE e DELETE em tb_itens_pedidos ✅")

Triggers criados: INSERT, UPDATE e DELETE em tb_itens_pedidos ✅


## 8. Testando: o banco atualiza `valor_total` sozinho

In [15]:
with Session(engine) as session:

    cli  = session.scalars(select(Cliente).limit(1)).first()
    prod = session.scalars(select(Produto).limit(1)).first()

    if not cli or not prod:
        print("Popule o banco com clientes e produtos antes de testar os triggers.")
    else:
        # Passo 1: cria pedido com valor_total = 0 (intencionalmente)
        novo_pedido = Pedido(
            cliente_id=cli.id,
            data_pedido=datetime.now(),
            status="Criado",
            valor_total=Decimal("0.00"),  # começa zerado — o trigger vai corrigir
        )
        session.add(novo_pedido)
        session.commit()
        session.refresh(novo_pedido)

        print(f"Antes de inserir itens  → valor_total: R$ {novo_pedido.valor_total}")

        # Passo 2: insere item — trigger dispara automaticamente!
        item = ItemPedido(
            pedido_id=novo_pedido.id,
            produto_id=prod.id,
            quantidade=3,
            preco_venda=Decimal("80.00"),  # 3 × 80 = R$ 240,00
        )
        session.add(item)
        session.commit()  # ← aqui o INSERT dispara o trigger no banco

        # session.refresh() sincroniza o objeto Python com o valor atual do banco
        # O trigger rodou no banco, mas o objeto em memória ainda tem o valor antigo!
        session.refresh(novo_pedido)

        print(f"Após inserir item       → valor_total: R$ {novo_pedido.valor_total}")
        print()
        print("✅ O banco atualizou valor_total automaticamente")
        print("   O código Python não tocou nesse campo!")

Antes de inserir itens  → valor_total: R$ 0.00
Após inserir item       → valor_total: R$ 240.00

✅ O banco atualizou valor_total automaticamente
   O código Python não tocou nesse campo!


> 💡 **Por que usar `session.refresh()`?**  
> O trigger rodou no banco, mas o objeto Python `novo_pedido` ainda tem o valor antigo em memória.  
> `session.refresh()` vai ao banco buscar o estado atual e sincroniza o objeto.

## 9. Inspecionando views e triggers no banco

In [16]:
with engine.connect() as conn:
    views = conn.execute(
        text("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name")
    ).fetchall()

    triggers = conn.execute(
        text("SELECT name, tbl_name FROM sqlite_master WHERE type='trigger' ORDER BY name")
    ).fetchall()

print("Views no banco:")
for v in views:
    print(f"   - {v.name}")

print("\n⚡ Triggers no banco:")
for t in triggers:
    print(f"   {t.name}  (tabela monitorada: {t.tbl_name})")

Views no banco:
   - vw_pedidos_resumo
   - vw_vendas_detalhadas

⚡ Triggers no banco:
   atualiza_total_delete  (tabela monitorada: tb_itens_pedidos)
   atualiza_total_insert  (tabela monitorada: tb_itens_pedidos)
   atualiza_total_update  (tabela monitorada: tb_itens_pedidos)


---

## Exercício: Usando IA para isso

**Prompt para gerar uma view a partir de uma pergunta:**
```
Com estas tabelas SQLite (tb_clientes, tb_pedidos, tb_itens_pedidos, tb_produtos),
crie uma VIEW chamada vw_receita_mensal que retorne:
ano, mês, total de pedidos, receita total e ticket médio.
Exclua pedidos com status 'Cancelado'.
Mostre também o DROP VIEW IF EXISTS antes do CREATE.
```

---

### Resposta:Código gerado pelo ChatGPT

In [17]:
with Session(engine) as session:

    # Remove a view se já existir
    session.execute(text("""
        DROP VIEW IF EXISTS vw_receita_mensal
    """))

    # Cria a view
    session.execute(text("""
        CREATE VIEW vw_receita_mensal AS
        SELECT
            strftime('%Y', p.data_pedido) AS ano,
            strftime('%m', p.data_pedido) AS mes,
            COUNT(DISTINCT p.id) AS total_pedidos,
            SUM(p.valor_total) AS receita_total,
            AVG(p.valor_total) AS ticket_medio
        FROM tb_pedidos p
        WHERE p.status != 'Cancelado'
        GROUP BY ano, mes
        ORDER BY ano, mes
    """))

    session.commit()

print("View vw_receita_mensal criada com sucesso!")

View vw_receita_mensal criada com sucesso!
